In [1]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.util import ngrams
from collections import defaultdict, Counter
import re

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [15]:
documents = [
    "One Piece is the greatest anime of all time.",
    "Messi is the best football player of all time.",
    "Football world cup 2022 final is thee greatest sports final ever."
]

query = "football player"
vocab = ["one", "piece", "is", "the", "greatest", "anime", "of", "all", "time", 
   "messi", "best", "football", "player", "world", "cup", "2022", 
   "final", "thee", "sports", "ever"]
corpus = ' '.join(documents)

1. Query Expansion

In [27]:
def queryExpansion(query):
    words = nltk.word_tokenize(query.lower())
    expanded = set(words)

    for word in words:
        synsets = wn.synsets(word)
        for syn in synsets:
            for lemma in syn.lemmas():
                synonym = lemma.name().replace('_', ' ').lower()
                expanded.add(synonym)

    return sorted(expanded)

print("Expanded Query:")
print(queryExpansion(query))

Expanded Query:
['actor', 'football', 'football game', 'histrion', 'instrumentalist', 'musician', 'participant', 'player', 'role player', 'thespian']


2. Spelling Correction

a) Edit Distance

In [38]:
def edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],  
                    dp[i][j - 1], 
                    dp[i - 1][j - 1]
                )
    return dp[m][n]

def correct_word(word, vocab):
    closest_word = min(vocab, key=lambda w: edit_distance(word, w))
    return closest_word

misspelled = "pfagalyer"
print("\n2A. Edit Distance Correction:")
print(f"Original: {misspelled} → Corrected: {correct_word(misspelled, vocab)}")


2A. Edit Distance Correction:
Original: pfagalyer → Corrected: player


b) K-Gram Index

In [37]:
def generate_kgrams(word, k=3):
    """Generate k-grams for a word, with start/end markers ($)."""
    word = f"${word}$"
    return [word[i:i+k] for i in range(len(word) - k + 1)]

def build_kgram_index(vocab, k=3):
    """Create a k-gram inverted index: k-gram → set of words containing it."""
    index = defaultdict(set)
    for word in vocab:
        for gram in generate_kgrams(word, k):
            index[gram].add(word)
    return index

def correct_kgram(word, kgram_index, k=3):
    """Find the closest word in vocab by counting shared k-grams."""
    grams = generate_kgrams(word, k)
    candidate_scores = Counter()
    
    for g in grams:
        for candidate in kgram_index.get(g, []):
            candidate_scores[candidate] += 1
    
    if candidate_scores:
        return candidate_scores.most_common(1)[0][0]
    return word

k_index = build_kgram_index(vocab, k=3)

misspelled = "pfagalyer"
corrected = correct_kgram(misspelled, k_index)
print("\n2B. K-Gram Correction:")
print(f"Original: {misspelled} → Corrected: {corrected}")


2B. K-Gram Correction:
Original: pfagalyer → Corrected: player


c) Context Sensitive Grammar

In [36]:
def train_bigram_model(corpus):
    """
    Build a bigram frequency model from the given text corpus.
    Returns a Counter with bigram tuples as keys.
    """
    tokens = nltk.word_tokenize(corpus.lower())
    bigram_list = list(ngrams(tokens, 2))
    return Counter(bigram_list)

def context_sensitive_correction(word_list, bigram_model, vocab):
    """
    Correct words based on bigram probabilities.
    word_list: list of words to correct
    bigram_model: trained bigram frequencies
    vocab: list of known words for edit distance correction
    """
    if not word_list:
        return []

    corrected = [word_list[0]] 
    for i in range(1, len(word_list)):
        prev_word = corrected[-1]
        current_word = word_list[i]
        
        candidates = [current_word, correct_word(current_word, vocab)]
        
        best_word = max(candidates, key=lambda w: bigram_model.get((prev_word, w), 0))
        corrected.append(best_word)
    
    return corrected

bigram_model = train_bigram_model(corpus)

misspelled_words = ["football", "plfsyd", "final"]
corrected_words = context_sensitive_correction(misspelled_words, bigram_model, vocab)

print("\n2C. Context-Sensitive Correction:")
print("Original Words: ", misspelled_words)
print("Corrected Words:", corrected_words)


2C. Context-Sensitive Correction:
Original Words:  ['football', 'plfsyd', 'final']
Corrected Words: ['football', 'player', 'final']


Query Language Interpreter

a) Single Word Query

In [35]:
def single_word_query(word, docs):
    return [doc for doc in docs if word.lower() in doc.lower()]

print(single_word_query("greatest", documents))

['One Piece is the greatest anime of all time.', 'Football world cup 2022 final is thee greatest sports final ever.']


b) Boolean Query

In [39]:
def boolean_query(query, docs):
    terms = query.lower().split()
    if "and" in terms:
        terms = [t for t in terms if t != "and"]
        return [doc for doc in docs if all(t in doc.lower() for t in terms)]
    elif "or" in terms:
        terms = [t for t in terms if t != "or"]
        return [doc for doc in docs if any(t in doc.lower() for t in terms)]
    elif "not" in terms:
        term = terms[terms.index("not")+1]
        return [doc for doc in docs if term not in doc.lower()]
    else:
        return [doc for doc in docs if any(t in doc.lower() for t in terms)]


print(boolean_query("one AND piece", documents))

['One Piece is the greatest anime of all time.']


c) Natural Language Query

In [41]:

def natural_language_query(query, docs):
    tokens = nltk.word_tokenize(query.lower())
    return [doc for doc in docs if any(t in doc.lower() for t in tokens)]
print(natural_language_query("Do you like one piece?", documents))

['One Piece is the greatest anime of all time.']


d) Structural Query

In [43]:
def structural_query(query, docs):
    m = re.match(r"(title|body):(\w+)", query.lower())
    if m:
        _, word = m.groups()
        return [doc for doc in docs if word in doc.lower()]
    return []

print(structural_query("title:player", documents))

['Messi is the best football player of all time.']
